# Activité Pratique N°5 — Agentic AI et MCP
## Partie A : LangChain

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Agent simple, mémoire, tools, tools prédéfinis, middlewares et chatbot RAG agentique, avec **Azure OpenAI** (déploiement `gpt-4.1-mini`).

## A.0 Préparation

- Charger le fichier `.env` avec `load_dotenv(override=True)`
- Créer **une seule fois** le modèle `llm = AzureChatOpenAI(...)` (avec `timeout=30`), réutilisé dans toute la partie A
- Tester la connexion avec `llm.invoke("Bonjour").content`

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

True

In [2]:
from langchain_openai import AzureChatOpenAI

AZURE = dict(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

llm = AzureChatOpenAI(
    **AZURE,
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    temperature=0,
    timeout=30,
)

In [3]:
llm.invoke("Bonjour, présente-toi en une phrase.").content

'Bonjour ! Je suis ChatGPT, une intelligence artificielle conçue pour t’aider à répondre à tes questions et à discuter de nombreux sujets.'

## A.1 Agent simple

- Importer `create_agent` depuis `langchain.agents`
- Créer un agent **sans tools** : `create_agent(model=llm, system_prompt="...")`
- L'appeler avec `agent.invoke({"messages": [HumanMessage("...")]})`
- Afficher la réponse : `response["messages"][-1].content`
- **Configuration du modèle** : faire varier `temperature` et `max_tokens` et observer l'effet

In [4]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=llm,
    system_prompt="Tu es un assistant pédagogique en intelligence artificielle. Réponds en français, de façon claire et concise.",
)

response = agent.invoke({"messages": [HumanMessage("Qu'est-ce qu'un agent IA ? Réponds en 3 phrases.")]})
display(Markdown(response["messages"][-1].content))

Un agent IA est un système informatique capable de percevoir son environnement grâce à des capteurs et d’agir sur celui-ci via des effecteurs. Il utilise des algorithmes pour prendre des décisions et atteindre des objectifs spécifiques. Ainsi, un agent IA peut apprendre, s’adapter et interagir de manière autonome dans son environnement.

In [5]:
# Ce que contient la réponse : la liste complète des messages de l'exécution
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Qu'est-ce qu'un agent IA ? Réponds en 3 phrases.
================================== Ai Message ==================================

Un agent IA est un système informatique capable de percevoir son environnement grâce à des capteurs et d’agir sur celui-ci via des effecteurs. Il utilise des algorithmes pour prendre des décisions et atteindre des objectifs spécifiques. Ainsi, un agent IA peut apprendre, s’adapter et interagir de manière autonome dans son environnement.


### Configuration et contrôle du modèle

- **`temperature`** : 0 = réponses stables et déterministes, plus haut = plus de variété
- **`max_tokens`** : limite la longueur de la réponse ; si la limite est atteinte, `finish_reason` vaut `length`

In [6]:
question = "Propose un nom original pour une startup d'IA agentique. Donne uniquement le nom."

for temperature in [0, 1.2]:
    modele = AzureChatOpenAI(**AZURE, azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
                             temperature=temperature, timeout=30)
    agent_temp = create_agent(model=modele)
    noms = [agent_temp.invoke({"messages": [HumanMessage(question)]})["messages"][-1].content for _ in range(3)]
    print(f"temperature={temperature} -> {noms}")

temperature=0 -> ['AgentiaNova', 'AgentiaNova', 'AgentiaNova']


temperature=1.2 -> ['Agentisys', 'AgeniusAI', 'CogniFlux']


In [7]:
for max_tokens in [20, 1000]:
    modele = AzureChatOpenAI(**AZURE, azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
                             temperature=0, max_tokens=max_tokens, timeout=30)
    agent_court = create_agent(model=modele)
    reponse = agent_court.invoke({"messages": [HumanMessage("Explique le pattern ReAct.")]})["messages"][-1]
    print(f"max_tokens={max_tokens} | finish_reason={reponse.response_metadata.get('finish_reason')} | "
          f"{len(reponse.content)} caractères")
    print(reponse.content[:200], "...\n")

max_tokens=20 | finish_reason=length | 79 caractères
Le pattern **ReAct** est une approche combinant **Reasoning** (raisonnement) et ...



max_tokens=1000 | finish_reason=stop | 2265 caractères
Le pattern **ReAct** est une approche utilisée en intelligence artificielle, notamment dans le domaine des agents conversationnels ou des systèmes d'IA capables de raisonner et d'agir de manière inter ...



## A.2 Mémoire de l'agent

- Importer `InMemorySaver` depuis `langgraph.checkpoint.memory`
- Créer l'agent avec `checkpointer=InMemorySaver()`
- Appeler l'agent avec `config={"configurable": {"thread_id": "1"}}`
- **Test** :
  - « Je m'appelle X » puis « Comment je m'appelle ? » avec le **même** `thread_id` → l'agent se souvient
  - Même question avec un **autre** `thread_id` → il ne sait pas

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(
    model=llm,
    system_prompt="Tu es un assistant sympathique. Réponds en français.",
    checkpointer=InMemorySaver(),
)

conversation_1 = {"configurable": {"thread_id": "1"}}
conversation_2 = {"configurable": {"thread_id": "2"}}

r = agent_memoire.invoke({"messages": [HumanMessage("Je m'appelle Mohamed et j'étudie à l'EMSI.")]}, config=conversation_1)
print("thread 1 :", r["messages"][-1].content)

thread 1 : Enchanté, Mohamed ! Comment se passent tes études à l'EMSI ? Y a-t-il quelque chose en particulier sur lequel tu aimerais de l’aide ?


In [9]:
r = agent_memoire.invoke({"messages": [HumanMessage("Comment je m'appelle et où j'étudie ?")]}, config=conversation_1)
print("thread 1 :", r["messages"][-1].content)

r = agent_memoire.invoke({"messages": [HumanMessage("Comment je m'appelle et où j'étudie ?")]}, config=conversation_2)
print("thread 2 :", r["messages"][-1].content)

thread 1 : Tu t'appelles Mohamed et tu étudies à l'EMSI.


thread 2 : Je suis désolé, mais je ne connais pas ton nom ni où tu étudies. Peux-tu me le dire ?


In [10]:
# L'état mémorisé pour le thread 1 (checkpoint)
etat = agent_memoire.get_state(conversation_1)
print(f"{len(etat.values['messages'])} messages mémorisés dans le thread 1")

4 messages mémorisés dans le thread 1


## A.3 Ajouter des tools à l'agent

- Écrire des fonctions avec le décorateur `@tool` (`from langchain.tools import tool`)
- Mettre des **annotations de type** et une **docstring claire** : c'est elle qui permet au LLM de choisir le tool
- Exemples : `get_weather(city)`, `get_employee_info(name)` (données factices)
- Créer l'agent : `create_agent(model=llm, tools=[...])`
- Afficher **tous** les `messages` de la réponse pour montrer les `tool_calls` et les `ToolMessage`

In [11]:
from langchain.tools import tool

METEO = {"casablanca": (19, "ensoleillé"), "rabat": (17, "nuageux"), "marrakech": (24, "ensoleillé"), "tanger": (15, "pluvieux")}
EMPLOYES = {"hassan": {"poste": "Data Scientist", "salaire": 23000, "anciennete": 12},
            "salma": {"poste": "Ingénieure MLOps", "salaire": 19500, "anciennete": 4}}


@tool
def get_weather(city: str) -> str:
    """Donne la météo actuelle (température en °C et ciel) d'une ville du Maroc."""
    print(f"  [tool] get_weather(city={city!r})")
    temperature, ciel = METEO.get(city.lower(), (20, "inconnu"))
    return f"{city} : {temperature} °C, ciel {ciel}"


@tool
def get_employee_info(name: str) -> dict:
    """Donne les informations RH d'un employé de l'entreprise : poste, salaire mensuel (MAD) et ancienneté (années)."""
    print(f"  [tool] get_employee_info(name={name!r})")
    return EMPLOYES.get(name.lower(), {"erreur": f"Aucun employé nommé {name}"})


agent_tools = create_agent(
    model=llm,
    tools=[get_weather, get_employee_info],
    system_prompt="Tu réponds en français en utilisant les tools fournis. N'invente aucune donnée.",
)

In [12]:
response = agent_tools.invoke({"messages": [HumanMessage("Quel temps fait-il à Marrakech et à Tanger ?")]})
for message in response["messages"]:
    message.pretty_print()

  [tool] get_weather(city='Marrakech')
  [tool] get_weather(city='Tanger')


================================ Human Message =================================

Quel temps fait-il à Marrakech et à Tanger ?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_RczZkaBzKSZYMizxClrSSOYm)
 Call ID: call_RczZkaBzKSZYMizxClrSSOYm
  Args:
    city: Marrakech
  get_weather (call_NyWXQJiWBPfQyliRVnaMdEtU)
 Call ID: call_NyWXQJiWBPfQyliRVnaMdEtU
  Args:
    city: Tanger
================================= Tool Message =================================
Name: get_weather

Marrakech : 24 °C, ciel ensoleillé
================================= Tool Message =================================
Name: get_weather

Tanger : 15 °C, ciel pluvieux
================================== Ai Message ==================================

À Marrakech, il fait 24 °C avec un ciel ensoleillé. À Tanger, il fait 15 °C avec un ciel pluvieux.


In [13]:
response = agent_tools.invoke({"messages": [HumanMessage("Quel est le salaire de Hassan, et qui a le plus d'ancienneté entre Hassan et Salma ?")]})
for message in response["messages"]:
    message.pretty_print()

  [tool] get_employee_info(name='Hassan')
  [tool] get_employee_info(name='Salma')


================================ Human Message =================================

Quel est le salaire de Hassan, et qui a le plus d'ancienneté entre Hassan et Salma ?
================================== Ai Message ==================================
Tool Calls:
  get_employee_info (call_xo8tPzz3do5r3eEcmZLCKChI)
 Call ID: call_xo8tPzz3do5r3eEcmZLCKChI
  Args:
    name: Hassan
  get_employee_info (call_VEDnp9PX3NkLBIkuPdrydSdm)
 Call ID: call_VEDnp9PX3NkLBIkuPdrydSdm
  Args:
    name: Salma
================================= Tool Message =================================
Name: get_employee_info

{"poste": "Data Scientist", "salaire": 23000, "anciennete": 12}
================================= Tool Message =================================
Name: get_employee_info

{"poste": "Ingénieure MLOps", "salaire": 19500, "anciennete": 4}
================================== Ai Message ==================================

Le salaire de Hassan est de 23 000 MAD par mois. En ce qui concerne l'ancienneté, Ha

## A.4 Tools prédéfinis

| Tool | Import | Remarque |
|---|---|---|
| **DuckDuckGo** | `from ddgs import DDGS` | Placé dans un `@tool`, aucune clé nécessaire |
| **Tavily** | `from langchain_tavily import TavilySearch` | `TavilySearch(max_results=3)`, utilise `TAVILY_API_KEY` |
| **PythonREPL** | `from langchain_experimental.tools import PythonREPLTool` | L'agent exécute du Python (calculs, graphiques) |

- Montrer un agent qui **combine plusieurs tools** et choisit le bon selon la question
- **Attention** : `PythonREPLTool` exécute réellement du code sur la machine

### DuckDuckGo

In [14]:
from ddgs import DDGS


@tool
def duckduckgo_search(query: str) -> list[dict]:
    """Recherche sur le web avec DuckDuckGo. Renvoie les 3 premiers résultats (titre, lien, extrait)."""
    print(f"  [tool] duckduckgo_search(query={query!r})")
    try:
        return [{"titre": r["title"], "lien": r["href"], "extrait": r["body"]}
                for r in DDGS().text(query, max_results=3)]
    except Exception as e:
        return [{"erreur": str(e)}]


duckduckgo_search.invoke({"query": "LangGraph framework"})[:1]

  [tool] duckduckgo_search(query='LangGraph framework')


[{'titre': 'Comparison of LangGraph, CrewAI, and Microsoft Agent Framework',
  'lien': 'https://grokipedia.com/page/Comparison_of_LangGraph_CrewAI_and_Microsoft_Agent_Framework',
  'extrait': 'As of February 23, 2026, this article compares three prominent frameworks for building AI agent workflows and multi-agent systems: LangGraph (from LangChain), CrewAI, and the newly released Microsoft Agent Framework. LangGraph is a graph-based frame…'}]

### Tavily

In [15]:
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(max_results=3)
resultat = tavily_search.invoke({"query": "Model Context Protocol Anthropic"})
[(r["title"], r["url"]) for r in resultat["results"]]

[('Model Context Protocol',
  'https://en.wikipedia.org/wiki/Model_Context_Protocol'),
 ('Anthropic’s Model Context Protocol (MCP): A Deep Dive for Developers | by Amanatullah | Medium',
  'https://medium.com/@amanatulla1606/anthropics-model-context-protocol-mcp-a-deep-dive-for-developers-1d3db39c9fdc'),
 ('Introducing the Model Context Protocol \\ Anthropic',
  'https://www.anthropic.com/news/model-context-protocol')]

### PythonREPLTool

In [16]:
from langchain_experimental.tools import PythonREPLTool

python_repl = PythonREPLTool()
print(python_repl.invoke("print(sum(i**2 for i in range(1, 101)))"))

C:\Users\user\AppData\Local\Temp\ipykernel_12492\111768688.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool
Python REPL can execute arbitrary code. Use with caution.


338350



### Agent qui combine plusieurs tools

L'agent choisit lui-même le tool adapté à chaque partie de la question.

In [17]:
agent_multi = create_agent(
    model=llm,
    tools=[tavily_search, duckduckgo_search, python_repl, get_weather],
    system_prompt=(
        "Tu es un assistant en français. Utilise tavily_search pour l'actualité, duckduckgo_search si Tavily échoue, "
        "Python_REPL pour tout calcul (avec print), get_weather pour la météo. Cite les liens des sources web."
    ),
)

In [18]:
response = agent_multi.invoke({"messages": [HumanMessage(
    "1) Calcule la factorielle de 20 et sa racine carrée. 2) Quel temps fait-il à Rabat ? "
    "3) Qu'est-ce que LangGraph ? (une phrase, avec la source)"
)]})
for message in response["messages"]:
    message.pretty_print()

  [tool] get_weather(city='Rabat')


================================ Human Message =================================

1) Calcule la factorielle de 20 et sa racine carrée. 2) Quel temps fait-il à Rabat ? 3) Qu'est-ce que LangGraph ? (une phrase, avec la source)
================================== Ai Message ==================================
Tool Calls:
  Python_REPL (call_w7nE0rwiSN2KAw78h9hvLsS0)
 Call ID: call_w7nE0rwiSN2KAw78h9hvLsS0
  Args:
    query: import math
fact_20 = math.factorial(20)
sqrt_fact_20 = math.sqrt(fact_20)
print(fact_20, sqrt_fact_20)
  get_weather (call_jVmQZOQ6v3MG0NfNv0YwVelS)
 Call ID: call_jVmQZOQ6v3MG0NfNv0YwVelS
  Args:
    city: Rabat
  tavily_search (call_jOXochHyGkdkYDF0v5xPNSVo)
 Call ID: call_jOXochHyGkdkYDF0v5xPNSVo
  Args:
    query: LangGraph
    search_depth: basic
    include_images: False
    time_range: year
    topic: general
================================= Tool Message =================================
Name: Python_REPL

2432902008176640000 1559776268.6284978

================

## A.5 Middlewares

Import depuis `langchain.agents.middleware`, puis `create_agent(..., middleware=[...])`

| Middleware | Élément LangChain | Démonstration |
|---|---|---|
| **dynamic_model** | `@wrap_model_call` | Changer de modèle / de réglages selon la longueur de la conversation |
| **dynamic_prompt** | `@dynamic_prompt` | Adapter le system prompt au contexte (ex. utilisateur expert ou débutant) |
| **tool_error_handling** | `@wrap_tool_call` | Rattraper l'exception d'un tool et renvoyer un `ToolMessage` d'erreur lisible |
| **Guardrails** | `PIIMiddleware`, `@before_agent`, `@after_agent` | Masquer e-mail / carte bancaire, bloquer un mot interdit, vérifier la réponse |
| **Human in the Loop** | `HumanInTheLoopMiddleware` + `checkpointer` | Pause avant un tool sensible, `result["__interrupt__"]`, reprise avec `Command(resume=...)` |

- **Bonus** : Structured Output avec un `BaseModel` Pydantic

### Hooks d'un middleware (cycle de vie de l'agent)

Un middleware peut intervenir à chaque étape : `before_agent` → `before_model` → `after_model` → `after_agent`.

In [19]:
import time
from langchain.agents.middleware import AgentMiddleware, AgentState


class HooksDemo(AgentMiddleware):
    def before_agent(self, state: AgentState, runtime):
        self.debut = time.time()
        print("before_agent")

    def before_model(self, state: AgentState, runtime):
        print(f"before_model  ({len(state['messages'])} messages envoyés au LLM)")

    def after_model(self, state: AgentState, runtime):
        print("after_model")

    def after_agent(self, state: AgentState, runtime):
        print(f"after_agent   (durée : {time.time() - self.debut:.2f} s)")


agent_hooks = create_agent(model=llm, tools=[get_weather], middleware=[HooksDemo()])
r = agent_hooks.invoke({"messages": [HumanMessage("Météo à Casablanca ?")]})
print("\nRéponse :", r["messages"][-1].content)

before_agent
before_model  (1 messages envoyés au LLM)


after_model
  [tool] get_weather(city='Casablanca')
before_model  (3 messages envoyés au LLM)


after_model
after_agent   (durée : 1.99 s)

Réponse : La météo à Casablanca est actuellement de 19 °C avec un ciel ensoleillé.


### dynamic_model

`@wrap_model_call` intercepte chaque appel au LLM et peut **remplacer le modèle** avant l'appel (`request.override(model=...)`).
Ici : un modèle « rapide » (réponses courtes) au début, puis un modèle « détaillé » quand la conversation dépasse 4 messages.

In [20]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

modele_rapide = AzureChatOpenAI(**AZURE, azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
                                temperature=0, max_tokens=120, timeout=30)
modele_detaille = AzureChatOpenAI(**AZURE, azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
                                  temperature=0.3, max_tokens=800, timeout=30)


@wrap_model_call
def dynamic_model(request: ModelRequest, handler) -> ModelResponse:
    nb_messages = len(request.messages)
    if nb_messages > 4:
        print(f"  [dynamic_model] {nb_messages} messages -> modèle détaillé")
        return handler(request.override(model=modele_detaille))
    print(f"  [dynamic_model] {nb_messages} messages -> modèle rapide")
    return handler(request.override(model=modele_rapide))


agent_dyn = create_agent(model=llm, middleware=[dynamic_model], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "dyn"}}
for question in ["Qu'est-ce que RAG ?", "Et l'agentic RAG ?", "Compare les deux de façon détaillée."]:
    r = agent_dyn.invoke({"messages": [HumanMessage(question)]}, config=config)
    print(f"Q : {question}\nR ({len(r['messages'][-1].content)} car.) : {r['messages'][-1].content[:150]}...\n")

  [dynamic_model] 1 messages -> modèle rapide


Q : Qu'est-ce que RAG ?
R (541 car.) : RAG peut désigner plusieurs choses selon le contexte, mais dans le domaine de l'intelligence artificielle et du traitement du langage naturel, **RAG**...

  [dynamic_model] 3 messages -> modèle rapide


Q : Et l'agentic RAG ?
R (547 car.) : L’**agentic RAG** est une extension ou une variante du modèle **RAG (Retrieval-Augmented Generation)** qui intègre une dimension **agentive**, c’est-à...

  [dynamic_model] 5 messages -> modèle détaillé


Q : Compare les deux de façon détaillée.
R (3865 car.) : Bien sûr ! Voici une comparaison détaillée entre **RAG (Retrieval-Augmented Generation)** classique et **Agentic RAG**, en mettant en lumière leurs pr...



### dynamic_prompt

Le system prompt est généré à chaque appel à partir du **contexte d'exécution** (`context_schema`).

In [21]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt


class Contexte(TypedDict):
    user_role: str


@dynamic_prompt
def prompt_selon_role(request: ModelRequest) -> str:
    role = request.runtime.context.get("user_role", "user")
    base = "Tu es un assistant en intelligence artificielle. Réponds en français en 4 phrases maximum."
    if role == "expert":
        return base + " L'utilisateur est expert : réponse technique, vocabulaire précis, formules si utile."
    if role == "beginner":
        return base + " L'utilisateur est débutant : explique simplement, avec une analogie, sans jargon."
    return base


agent_role = create_agent(model=llm, middleware=[prompt_selon_role], context_schema=Contexte)
for role in ["expert", "beginner"]:
    r = agent_role.invoke({"messages": [HumanMessage("Explique l'attention dans les Transformers.")]},
                          context={"user_role": role})
    print(f"--- {role} ---")
    display(Markdown(r["messages"][-1].content))

--- expert ---


L'attention dans les Transformers est un mécanisme qui calcule un poids d'importance entre chaque paire de tokens via un produit scalaire entre requêtes (Q) et clés (K), normalisé par une softmax : \( \text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V \). Cela permet au modèle de pondérer dynamiquement les valeurs (V) en fonction du contexte, capturant ainsi les dépendances globales dans la séquence. Le facteur \( \sqrt{d_k} \) stabilise les gradients en évitant des valeurs trop grandes dans le produit scalaire. Cette opération est parallélisable et constitue la base des architectures Transformer.

--- beginner ---


L'attention dans les Transformers, c’est comme si tu étais en train de lire une phrase et que tu décidais de te concentrer sur certains mots importants pour mieux comprendre le sens. Imagine que chaque mot a une lampe de poche, et l’attention éclaire plus fort les mots utiles pour deviner le prochain mot. Cela aide le modèle à ne pas se perdre dans toute la phrase, mais à choisir ce qui compte vraiment. C’est un peu comme écouter une conversation et prêter plus attention aux phrases clés.

### tool_error_handling

Sans gestion, une exception dans un tool arrête l'agent. Avec `@wrap_tool_call`, l'erreur est transformée en `ToolMessage` : le LLM la lit et peut corriger son appel.

In [22]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage


@tool
def diviser(a: float, b: float) -> float:
    """Divise a par b."""
    print(f"  [tool] diviser({a}, {b})")
    return a / b


@wrap_tool_call
def tool_error_handling(request, handler):
    try:
        return handler(request)
    except Exception as e:
        print(f"  [middleware] erreur rattrapée : {type(e).__name__}: {e}")
        return ToolMessage(
            content=f"Erreur du tool : {type(e).__name__} ({e}). Corrige les arguments ou explique le problème à l'utilisateur.",
            tool_call_id=request.tool_call["id"],
        )


agent_erreurs = create_agent(model=llm, tools=[diviser], middleware=[tool_error_handling])
r = agent_erreurs.invoke({"messages": [HumanMessage("Calcule 10 divisé par 0 avec le tool, puis 10 divisé par 4.")]})
for message in r["messages"]:
    message.pretty_print()

  [tool] diviser(10.0, 0.0)
  [middleware] erreur rattrapée : ZeroDivisionError: float division by zero
  [tool] diviser(10.0, 4.0)


================================ Human Message =================================

Calcule 10 divisé par 0 avec le tool, puis 10 divisé par 4.
================================== Ai Message ==================================
Tool Calls:
  diviser (call_GPLSYOIsEDg9CfRvwYVZwZQp)
 Call ID: call_GPLSYOIsEDg9CfRvwYVZwZQp
  Args:
    a: 10
    b: 0
  diviser (call_44yNRVAnHCjgF196zQnzWOco)
 Call ID: call_44yNRVAnHCjgF196zQnzWOco
  Args:
    a: 10
    b: 4
================================= Tool Message =================================

Erreur du tool : ZeroDivisionError (float division by zero). Corrige les arguments ou explique le problème à l'utilisateur.
================================= Tool Message =================================
Name: diviser

2.5
================================== Ai Message ==================================

La division de 10 par 0 n'est pas possible car la division par zéro est indéfinie en mathématiques. En revanche, 10 divisé par 4 est égal à 2,5.


### Guardrails

**1. Détection des PII** (`PIIMiddleware`) : e-mail masqué (`redact`), carte bancaire masquée (`mask`), clé d'API bloquée (`block`).

In [23]:
from langchain.agents.middleware import PIIMiddleware
from langchain.agents.middleware.pii import PIIDetectionError


@tool
def payer_par_carte(carte: str) -> str:
    """Effectue un paiement avec la carte bancaire fournie."""
    print(f"  [tool] payer_par_carte reçoit : {carte}")
    return f"Paiement accepté avec la carte {carte}"


@tool
def envoyer_email(adresse: str) -> str:
    """Envoie un e-mail de confirmation à l'adresse fournie."""
    print(f"  [tool] envoyer_email reçoit : {adresse}")
    return f"E-mail envoyé à {adresse}"


agent_pii = create_agent(
    model=llm,
    tools=[payer_par_carte, envoyer_email],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]{32}", strategy="block", apply_to_input=True),
    ],
)

r = agent_pii.invoke({"messages": [HumanMessage("Envoie un e-mail à etudiant@emsi.ma et paie avec la carte 5105-1051-0510-5100.")]})
print("Message vu par le LLM :", r["messages"][0].content)
print("Réponse :", r["messages"][-1].content)

  [tool] payer_par_carte reçoit : ****-****-****-5100
  [tool] envoyer_email reçoit : [REDACTED_EMAIL]


Message vu par le LLM : Envoie un e-mail à [REDACTED_EMAIL] et paie avec la carte ****-****-****-5100.
Réponse : Le paiement avec la carte ****-****-****-5100 a été effectué et un e-mail a été envoyé à l'adresse indiquée. Y a-t-il autre chose que je puisse faire pour vous ?


In [24]:
try:
    agent_pii.invoke({"messages": [HumanMessage("Utilise ma clé sk-" + "a1B2" * 8 + " pour te connecter.")]})
except PIIDetectionError as e:
    print("Requête bloquée :", e)

Requête bloquée : Detected 1 instance(s) of api_key in text content


**2. Garde-fou déterministe avant l'agent** (`@before_agent`) : une requête contenant un mot interdit est arrêtée **avant** tout appel au LLM.

In [25]:
from typing import Any
from langchain.agents.middleware import before_agent, after_agent
from langgraph.runtime import Runtime

MOTS_INTERDITS = ["hack", "pirater", "malware", "exploit"]


@before_agent(can_jump_to=["end"])
def filtre_contenu(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    texte = state["messages"][-1].content.lower()
    for mot in MOTS_INTERDITS:
        if mot in texte:
            print(f"  [guardrail] mot interdit détecté : {mot!r} -> fin immédiate")
            return {"messages": [{"role": "assistant", "content": "Je ne peux pas traiter cette demande."}],
                    "jump_to": "end"}
    return None


agent_filtre = create_agent(model=llm, tools=[tavily_search], middleware=[filtre_contenu])
for question in ["Comment pirater le wifi de mon voisin ?", "Donne une définition courte du Model Context Protocol (MCP) en IA."]:
    r = agent_filtre.invoke({"messages": [HumanMessage(question)]})
    print(f"Q : {question}\nR : {r['messages'][-1].content[:200]}\n")

  [guardrail] mot interdit détecté : 'pirater' -> fin immédiate
Q : Comment pirater le wifi de mon voisin ?
R : Je ne peux pas traiter cette demande.



Q : Donne une définition courte du Model Context Protocol (MCP) en IA.
R : Le Model Context Protocol (MCP) en IA est un cadre ou protocole qui définit comment un modèle d'intelligence artificielle doit gérer, interpréter et utiliser le contexte d'une interaction ou d'une don



**3. Garde-fou fondé sur un modèle après l'agent** (`@after_agent`) : un second LLM juge la réponse finale `SAFE` / `UNSAFE`.

In [26]:
from langchain.messages import AIMessage

modele_securite = AzureChatOpenAI(**AZURE, azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"), temperature=0, timeout=30)


@after_agent(can_jump_to=["end"])
def verification_securite(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    derniere = state["messages"][-1]
    if not isinstance(derniere, AIMessage):
        return None
    verdict = modele_securite.invoke(
        "Cette réponse est-elle sûre et appropriée pour un public étudiant ? Réponds uniquement SAFE ou UNSAFE.\n\n"
        f"Réponse : {derniere.content}"
    ).content.strip()
    print(f"  [guardrail après agent] verdict : {verdict}")
    if "UNSAFE" in verdict:
        derniere.content = "Je ne peux pas fournir cette réponse."
    return None


agent_securise = create_agent(model=llm, middleware=[verification_securite])
r = agent_securise.invoke({"messages": [HumanMessage("Donne 3 conseils pour bien réviser un examen d'IA.")]})
print(r["messages"][-1].content)

  [guardrail après agent] verdict : SAFE
Bien sûr ! Voici trois conseils pour bien réviser un examen d’intelligence artificielle (IA) :

1. **Comprendre les concepts clés**  
   Plutôt que d’apprendre par cœur, assure-toi de bien comprendre les notions fondamentales comme les algorithmes d’apprentissage supervisé et non supervisé, les réseaux de neurones, le traitement du langage naturel, ou encore les méthodes d’optimisation. Essaie d’expliquer ces concepts avec tes propres mots ou à quelqu’un d’autre.

2. **Pratiquer avec des exercices et des projets**  
   L’IA est une discipline très pratique. Mets-toi à coder des algorithmes simples (par exemple, un perceptron, une régression linéaire, ou un arbre de décision) et utilise des bibliothèques comme scikit-learn ou TensorFlow. Résoudre des problèmes concrets t’aidera à mieux assimiler les notions théoriques.

3. **Revoir les cours et les ressources complémentaires**  
   Relis tes notes de cours, mais aussi consulte des ressources en l

### Human in the Loop

L'agent s'arrête **avant** d'exécuter un tool sensible et attend la décision d'un humain (`approve`, `edit` ou `reject`).
Le `checkpointer` est obligatoire : il conserve l'état pendant la pause.

In [27]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command


@tool
def supprimer_fichier(nom: str) -> str:
    """Supprime définitivement un fichier du serveur."""
    print(f"  [tool] supprimer_fichier({nom!r}) EXÉCUTÉ")
    return f"Fichier {nom} supprimé"


agent_hitl = create_agent(
    model=llm,
    tools=[envoyer_email, supprimer_fichier],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"envoyer_email": True, "supprimer_fichier": True})],
    checkpointer=InMemorySaver(),
)

config_hitl = {"configurable": {"thread_id": "hitl-1"}}
resultat = agent_hitl.invoke({"messages": [HumanMessage("Envoie un e-mail à prof@emsi.ma")]}, config=config_hitl)
print("Interruption :", resultat["__interrupt__"][0].value["action_requests"])

Interruption : [{'name': 'envoyer_email', 'args': {'adresse': 'prof@emsi.ma'}, 'description': "Tool execution requires approval\n\nTool: envoyer_email\nArgs: {'adresse': 'prof@emsi.ma'}"}]


In [28]:
# L'humain approuve : le tool s'exécute et l'agent termine
resultat = agent_hitl.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config_hitl)
print(resultat["messages"][-1].content)

  [tool] envoyer_email reçoit : prof@emsi.ma


L'e-mail a été envoyé à prof@emsi.ma. Souhaitez-vous que je vous aide avec autre chose ?


In [29]:
# Deuxième cas : l'humain refuse la suppression
config_hitl_2 = {"configurable": {"thread_id": "hitl-2"}}
resultat = agent_hitl.invoke({"messages": [HumanMessage("Supprime le fichier notes_examen.xlsx")]}, config=config_hitl_2)
print("Interruption :", resultat["__interrupt__"][0].value["action_requests"])

resultat = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Suppression refusée par l'administrateur."}]}),
    config=config_hitl_2,
)
print("Réponse :", resultat["messages"][-1].content)

Interruption : [{'name': 'supprimer_fichier', 'args': {'nom': 'notes_examen.xlsx'}, 'description': "Tool execution requires approval\n\nTool: supprimer_fichier\nArgs: {'nom': 'notes_examen.xlsx'}"}]


Réponse : La suppression du fichier "notes_examen.xlsx" a été refusée par l'administrateur. Je ne peux donc pas effectuer cette action. Y a-t-il autre chose que je puisse faire pour vous ?


### Bonus : Structured Output

`response_format` force l'agent à produire un objet conforme à un schéma Pydantic.

In [30]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy


class Contact(BaseModel):
    nom: str
    email: str
    telephone: str = Field(description="Numéro au format international")


agent_extraction = create_agent(model=llm, response_format=ToolStrategy(Contact))
r = agent_extraction.invoke({"messages": [HumanMessage(
    "Extrais le contact : BENHAYOUN SADAFI Mohamed, joignable à contact@exemple.ma ou au 06 12 34 56 78 (Maroc).")]})
r["structured_response"]

Contact(nom='BENHAYOUN SADAFI Mohamed', email='contact@exemple.ma', telephone='+212612345678')

## A.6 Chatbot RAG agentique

1. **Charger** un document PDF avec `PyPDFLoader` (`langchain_community.document_loaders`)
2. **Découper** avec `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)`
3. **Embeddings** avec `AzureOpenAIEmbeddings(...)`
4. **Indexer** dans `FAISS.from_documents(chunks, embeddings)`
5. **Créer le tool** : `create_retriever_tool(vector_store.as_retriever(), "nom", "description")`
6. **Créer l'agent** avec ce tool, une mémoire (`InMemorySaver`) et un system prompt qui demande de **citer les sources**
7. **Tester** avec 3 ou 4 questions qui se suivent : l'agent cherche dans le document **et** se souvient de la conversation

Document indexé : le support de cours **« Agentic AI et MCP »** (116 pages).

In [31]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

PDF_COURS = Path("Supports de cours/Annonce - Agentic AI et MCP - Langchain et Langgraph/Agentic AI et MCP.pdf")
pages = PyPDFLoader(str(PDF_COURS)).load()
print(f"{len(pages)} pages chargées | page 4 : {pages[3].page_content[:150]!r}")

C:\Users\user\AppData\Local\Temp\ipykernel_12492\124520418.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


116 pages chargées | page 4 : 'AI Agents\nFonctionnement :\n• Un agent opère généralement selon un design pattern nommé React (Reasoning and \nAction). Ce cycle comprend plusieurs étap'


In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents([p for p in pages if p.page_content.strip()])
print(f"{len(chunks)} chunks | métadonnées : {chunks[0].metadata}")

137 chunks | métadonnées : {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-11-18T10:03:34+00:00', 'title': 'Présentation PowerPoint', 'author': 'Mohamed Youssfi', 'moddate': '2026-11-18T10:03:34+00:00', 'source': 'Supports de cours\\Annonce - Agentic AI et MCP - Langchain et Langgraph\\Agentic AI et MCP.pdf', 'total_pages': 116, 'page': 0, 'page_label': '1'}


In [33]:
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = AzureOpenAIEmbeddings(**AZURE, azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"), chunk_size=64)

vector_store = FAISS.from_documents(chunks, embeddings)
# Sauvegarde pour la partie B (un seul fichier : FAISS ne gère pas les accents du chemin « Activité Pratique N°5 »)
Path("index_faiss_cours.bin").write_bytes(vector_store.serialize_to_bytes())
print("Vecteurs indexés :", vector_store.index.ntotal)

Vecteurs indexés : 137


In [34]:
# Test du retriever seul
for doc in vector_store.similarity_search("Qu'est-ce que le Human in the loop ?", k=3):
    print(f"p. {doc.metadata['page_label']} | {doc.page_content[:120]!r}")

p. 101 | 'Human In The Loop'
p. 102 | 'Human In The Loop'
p. 96 | 'Human In The Loop\nLLM\nState\nRecipient = ..\nSubject = ..\nIntent = ..\nDraft = ..\nApproved = ..\nFeedback = ..\nGraph'


In [35]:
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import create_retriever_tool

outil_cours = create_retriever_tool(
    vector_store.as_retriever(search_kwargs={"k": 4}),
    name="rechercher_cours_agentic_ai",
    description=(
        "Recherche des passages dans le support de cours « Agentic AI et MCP » (LangChain, LangGraph, middlewares, "
        "guardrails, RAG, MCP). À utiliser pour toute question sur le cours."
    ),
    document_prompt=PromptTemplate.from_template("[p. {page_label}] {page_content}"),
)

chatbot_rag = create_agent(
    model=llm,
    tools=[outil_cours],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Tu es un assistant pédagogique sur le cours « Agentic AI et MCP ». Réponds en français.\n"
        "- Pour toute question sur le cours, utilise d'abord rechercher_cours_agentic_ai (tu peux reformuler la requête).\n"
        "- Appuie-toi uniquement sur les passages retrouvés et cite les pages sous la forme [p. N].\n"
        "- Si l'information n'est pas dans le cours, dis-le.\n"
        "- Pour les salutations ou les questions sur la conversation, réponds sans chercher."
    ),
)

In [36]:
def chatter(question: str, thread_id: str = "rag-1"):
    r = chatbot_rag.invoke({"messages": [HumanMessage(question)]}, config={"configurable": {"thread_id": thread_id}})
    dernier_humain = max(i for i, m in enumerate(r["messages"]) if isinstance(m, HumanMessage))
    appels = [tc["args"]["query"] for m in r["messages"][dernier_humain:] if isinstance(m, AIMessage) for tc in m.tool_calls]
    print(f"Vous : {question}")
    if appels:
        print(f"  (recherches : {appels})")
    display(Markdown(r["messages"][-1].content))
    print("-" * 80)


chatter("Bonjour ! Je prépare mon TP sur les agents.")
chatter("Quelles sont les étapes du pattern ReAct ?")
chatter("Et comment LangChain permet d'ajouter une validation humaine avant un tool ?")
chatter("Résume en une phrase ce dont on a parlé depuis le début.")

Vous : Bonjour ! Je prépare mon TP sur les agents.


Bonjour ! N'hésite pas à me poser tes questions sur les agents ou sur le cours « Agentic AI et MCP ». Je suis là pour t'aider. Que souhaites-tu savoir ou préparer pour ton TP ?

--------------------------------------------------------------------------------


Vous : Quelles sont les étapes du pattern ReAct ?
  (recherches : ['étapes du pattern ReAct'])


Le pattern ReAct (Reasoning and Action) comprend plusieurs étapes clés dans le fonctionnement d'un agent :

1. Raisonnement : Le cerveau (LLM) analyse la situation et détermine l'action optimale à prendre.
2. Action : L'agent utilise un outil spécifique pour interagir avec l'extérieur.
3. Observation : L'agent collecte les résultats de son action (feedback ou contexte) et renvoie ces observations à son cerveau pour raisonner à nouveau.
4. Mémoire : Pour pallier le caractère "stateless" des LLM, l'agent conserve un historique de la conversation afin de maintenir la cohérence au fil des échanges.

Ce cycle se répète jusqu'à ce que l'agent dispose de tout le contexte nécessaire pour atteindre son but [p. 4].

--------------------------------------------------------------------------------


... (2 outputs left out)

--------------------------------------------------------------------------------


Vous : Résume en une phrase ce dont on a parlé depuis le début.


Nous avons vu que le pattern ReAct décrit un cycle de raisonnement, action, observation et mémoire pour les agents, et que LangChain permet d'ajouter une validation humaine avant l'exécution d'outils sensibles grâce à un middleware Human-in-the-loop.

--------------------------------------------------------------------------------


In [37]:
# Une question hors du cours : le chatbot doit le dire au lieu d'inventer
chatter("Quel est le prix d'un abonnement Azure OpenAI ?")

Vous : Quel est le prix d'un abonnement Azure OpenAI ?
  (recherches : ['prix abonnement Azure OpenAI'])


Le cours « Agentic AI et MCP » ne contient pas d'information sur le prix d'un abonnement Azure OpenAI.

--------------------------------------------------------------------------------


**Conclusion de la partie A :** `create_agent` construit un agent ReAct (LLM + tools) ; la mémoire est un `checkpointer`, et les middlewares interviennent autour des appels au modèle et aux tools pour adapter le modèle, le prompt, gérer les erreurs, filtrer les entrées/sorties et insérer une validation humaine. Le RAG devient agentique quand la recherche documentaire est un **tool** que l'agent décide d'appeler.